# herbert large full emotion — pełny fine-tune (Kaggle T4)

In [ ]:
# transformers <5.2 — w 5.2 usunięto warmup_ratio z TrainingArguments.
# herbert_large_epochs padł na tym 10.08.2026). Górne ograniczenie utrzymuje
# recepturę identyczną z wcześniejszymi runami tej kampanii.
!pip install -q -U "transformers>=4.44,<5.2" "datasets>=2.20" accelerate 2>/dev/null
import torch, transformers
print(transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import (f1_score, hamming_loss, jaccard_score, accuracy_score, precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
warnings.filterwarnings("ignore")
RANDOM_STATE=42; torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
OUT="/kaggle/working"
MODEL_NAME="allegro/herbert-large-cased"; MAX_LEN, EPOCHS, BATCH, LR = 128, 3, 16, 2e-5

In [ ]:
def find_csv(n):
    h=glob.glob(f"/kaggle/input/**/{n}",recursive=True)
    if not h: raise FileNotFoundError(f"{n} — dołącz dataset pl-emotion-processed")
    return h[0]
tw_train=pd.read_csv(find_csv("twitteremo_train.csv")); tw_val=pd.read_csv(find_csv("twitteremo_val.csv")); tw_test=pd.read_csv(find_csv("twitteremo_test.csv"))
for d in (tw_train,tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw_val[EMOTIONS].values,tw_test[EMOTIONS].values
print("train",len(tw_train))

In [ ]:
def evaluate(yt,yp):
    return {"f1_macro":f1_score(yt,yp,average="macro",zero_division=0),"f1_micro":f1_score(yt,yp,average="micro",zero_division=0),
            "f1_weighted":f1_score(yt,yp,average="weighted",zero_division=0),"precision_macro":precision_score(yt,yp,average="macro",zero_division=0),
            "recall_macro":recall_score(yt,yp,average="macro",zero_division=0),"hamming_loss":hamming_loss(yt,yp),
            "jaccard_macro":jaccard_score(yt,yp,average="macro",zero_division=0),"subset_accuracy":accuracy_score(yt,yp)}
def find_optimal_thresholds(yt,yp,labels):
    thr=np.full(len(labels),0.5)
    for i in range(len(labels)):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            f=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[i]=bt
    return thr
def f1_macro_ci(yt,yp,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); n=len(yt); base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi

In [ ]:
pos=tw_train[EMOTIONS].values.sum(0); neg=len(tw_train)-pos
POS_WEIGHT=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)
class WeightedTrainer(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),pos_weight=self.pw.to(out.logits.device))
        return (loss,out) if return_outputs else loss

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
def to_ds(df):
    d=Dataset.from_dict({"text":df["tekst"].tolist(),"labels":df[EMOTIONS].values.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns=["text"])
ds_train,ds_val,ds_test=to_ds(tw_train),to_ds(tw_val),to_ds(tw_test)
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
args=TrainingArguments(output_dir=f"{OUT}/ckpt",eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,
    load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,gradient_accumulation_steps=2,gradient_checkpointing=True,num_train_epochs=EPOCHS,
    learning_rate=LR,warmup_ratio=0.1,weight_decay=0.01,fp16=True,logging_steps=100,report_to="none",seed=RANDOM_STATE)
cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
trainer=WeightedTrainer(model=model,args=args,train_dataset=ds_train,eval_dataset=ds_val,data_collator=DataCollatorWithPadding(tok),
    compute_metrics=cm,pos_weight=POS_WEIGHT,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

In [ ]:
p_val=expit(trainer.predict(ds_val).predictions); p_test=expit(trainer.predict(ds_test).predictions)
thr=find_optimal_thresholds(y_val,p_val,EMOTIONS); pred=(p_test>=thr).astype(int)
m=evaluate(y_test,pred); base,lo,hi=f1_macro_ci(y_test,pred)
m.update({"model":MODEL_NAME.split("/")[-1]+"-full","ci_low":round(lo,3),"ci_high":round(hi,3)})
pd.DataFrame([m]).to_csv(f"{OUT}/result.csv",index=False); np.save(f"{OUT}/herbert_large_full_proba_test.npy",p_test); np.save(f"{OUT}/herbert_large_full_proba_val.npy",p_val)  # pary do ensemble (notebook 15)
print(f"F1-Macro={m['f1_macro']:.3f}  [{lo:.3f},{hi:.3f}]  F1-Micro={m['f1_micro']:.3f}")
pd.DataFrame([m])[["model","f1_macro","ci_low","ci_high","f1_micro","jaccard_macro","hamming_loss"]]